# Step 5 — Feature Selection

Entrena un LightGBM rápido para ranking de importancia y selecciona top-N features.

**Estrategia para evitar OOM:**
- Train: muestra estratificada (todos los positivos + ~500K negativos)
- Val: muestra más pequeña para early stopping

**Nota:** Las fire_lag y spatial features no están en el dataset actual. Se construirán más adelante y se añadirán al pipeline.

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / 'src'))

from project_config import init_notebook
config = init_notebook()

import utils
logger = utils.init_logger('info')

In [ ]:
import json
import pandas as pd
import pyarrow.parquet as pq
from analytics import df_split, get_features
from models.stacking.base_learners import LGBMBaseLearner

project_folder = Path(config['project_folder'])
processed_dir = project_folder / config['data']['processed']['local_path']
target_col = config['model']['objective_column']

print(f'Target: {target_col}')
print(f'Fixed features (siempre incluir): {config["model"]["fixed_features"]}')

## 5.1 Cargar muestra estratificada del train

In [ ]:
from tqdm.auto import tqdm

NEG_LIMIT = 500_000  # Muestra más grande para mejor ranking

pf_train = pq.ParquetFile(processed_dir / 'train.parquet')

positives = []
negatives = []

for batch in tqdm(pf_train.iter_batches(batch_size=2_000_000), 
                  total=pf_train.metadata.num_row_groups,
                  desc='Sampling train'):
    df = batch.to_pandas()
    pos = df[df[target_col] == True]
    neg = df[df[target_col] == False]
    
    positives.append(pos)
    current_neg = sum(len(n) for n in negatives)
    if current_neg < NEG_LIMIT:
        remaining = NEG_LIMIT - current_neg
        negatives.append(neg.sample(n=min(len(neg), remaining), random_state=42))

train_sample = pd.concat(positives + negatives, ignore_index=True)
del positives, negatives

n_pos = train_sample[target_col].sum()
print(f'Train sample: {len(train_sample):,} filas ({n_pos:,} positivos)')

In [ ]:
# Muestra del val para early stopping (más pequeña)
pf_val = pq.ParquetFile(processed_dir / 'val.parquet')

val_positives = []
val_negatives = []
VAL_NEG_LIMIT = 50_000

for batch in pf_val.iter_batches(batch_size=2_000_000):
    df = batch.to_pandas()
    pos = df[df[target_col] == True]
    neg = df[df[target_col] == False]
    
    val_positives.append(pos)
    current_neg = sum(len(n) for n in val_negatives)
    if current_neg < VAL_NEG_LIMIT:
        remaining = VAL_NEG_LIMIT - current_neg
        val_negatives.append(neg.sample(n=min(len(neg), remaining), random_state=42))

val_sample = pd.concat(val_positives + val_negatives, ignore_index=True)
del val_positives, val_negatives

n_val_pos = val_sample[target_col].sum()
print(f'Val sample: {len(val_sample):,} filas ({n_val_pos:,} positivos)')

## 5.2 Preparar datos para LightGBM

In [ ]:
X_train, y_train = df_split(train_sample, target_col)
X_val, y_val = df_split(val_sample, target_col)

print(f'Features: {X_train.shape[1]}')
print(f'Feature names: {list(X_train.columns)}')

## 5.3 Entrenar LightGBM para feature importance

In [ ]:
# LightGBM con parámetros para ranking rápido
params = {
    'n_estimators': 500,
    'learning_rate': 0.1,
    'max_depth': 6,
    'num_leaves': 31,
    'scale_pos_weight': 10,  # Balanceo moderado para la muestra
    'random_state': 42,
}

learner = LGBMBaseLearner(params=params)
learner.fit(X_train, y_train, X_val, y_val)

print('\nEntrenamiento completado')

In [ ]:
# Obtener importancia de features
importance_dict = learner.get_feature_importance()

importance_df = pd.DataFrame({
    'feature_name': list(importance_dict.keys()),
    'importance': list(importance_dict.values())
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('Top 20 features por importancia:')
print(importance_df.head(20).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

# Visualizar importancia
plt.figure(figsize=(10, 8))
top_n = min(20, len(importance_df))
plt.barh(importance_df['feature_name'].head(top_n)[::-1], 
         importance_df['importance'].head(top_n)[::-1])
plt.xlabel('Importance')
plt.title('Feature Importance (LightGBM)')
plt.tight_layout()
plt.show()

## 5.4 Seleccionar features

In [ ]:
# Con 24 features actuales, no hay mucho que recortar
# Pero seguimos el proceso para cuando se agreguen fire_lag y spatial features

fixed_features = config['model']['fixed_features']
print(f'Fixed features (siempre incluir): {fixed_features}')

# Seleccionar todas las features actuales (24)
# En el futuro, cuando haya más features, usar get_features con features_amount
selected_features = list(X_train.columns)

# Asegurar que fixed_features estén al inicio
for f in fixed_features:
    if f in selected_features:
        selected_features.remove(f)
selected_features = fixed_features + selected_features

print(f'\nFeatures seleccionadas: {len(selected_features)}')
print(selected_features)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# Evaluar en val (tomar columna 1 = prob de clase positiva)
y_pred_proba = learner.predict_proba(X_val)[:, 1]
y_pred = (y_pred_proba > 0.5).astype(int)

roc_auc = roc_auc_score(y_val, y_pred_proba)
pr_auc = average_precision_score(y_val, y_pred_proba)
f1 = f1_score(y_val, y_pred)

print('Métricas en val sample (modelo de ranking, NO el modelo final):')
print(f'  AUC-ROC: {roc_auc:.4f}')
print(f'  AUC-PR:  {pr_auc:.4f}')
print(f'  F1:      {f1:.4f}')

## 5.5 Evaluación rápida del modelo de ranking

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# Evaluar en val
y_pred_proba = learner.predict_proba(X_val)
y_pred = (y_pred_proba > 0.5).astype(int)

roc_auc = roc_auc_score(y_val, y_pred_proba)
pr_auc = average_precision_score(y_val, y_pred_proba)
f1 = f1_score(y_val, y_pred)

print('Métricas en val sample (modelo de ranking, NO el modelo final):')
print(f'  AUC-ROC: {roc_auc:.4f}')
print(f'  AUC-PR:  {pr_auc:.4f}')
print(f'  F1:      {f1:.4f}')

## Resumen

- LightGBM entrenado en muestra estratificada para ranking de features
- Todas las 24 features actuales seleccionadas (pocas para recortar)
- Features guardadas en `data/processed/selected_features.json`

**Nota sobre fire_lag features:** Las features temporales de lag (`fire_lag_1d`, etc.) y espaciales (`fire_neighbors_*`) no están en el dataset actual. Se construirán e integrarán en una fase posterior.

**Siguiente paso**: `step6_model_selection.ipynb`